In [0]:
# from databricks.sdk import WorkspaceClient

# w = WorkspaceClient()

# for ep in w.serving_endpoints.list():
#     if ep.name.startswith("databricks-"):
#         print(ep.name)

In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
import mlflow.deployments

context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = context.apiUrl().get()
token = context.apiToken().get()

w = WorkspaceClient(host=host, token=token)
deploy_client = mlflow.deployments.get_deploy_client("databricks")

index_name = "dbr_dev_trial.artemzharkov10_gold.knowledge_base_index"
question = "What is the braking distance on wet asphalt and what is hydroplaning?"
print(f"Text: {question}\n")

In [0]:
search_results = w.vector_search_indexes.query_index(
    index_name=index_name,
    query_text=question,
    columns=["text_content"],
    num_results=3
)

context_chunks = []
if search_results.result and search_results.result.data_array:
    for row in search_results.result.data_array:
        context_chunks.append(row[0]) 

context_text = "\n\n---\n\n".join(context_chunks)
    

if context_text:  
    system_prompt = f"""You are a helpful road safety assistant. 
    Answer the user's question based ONLY on the following context. 
    If the context does not contain the answer, say "I don't have enough information".
    
    Context:
    {context_text}
    """

    llm_response = deploy_client.predict(
        endpoint="databricks-meta-llama-3-3-70b-instruct",
        inputs={
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": question}
            ],
            "max_tokens": 512,
            "temperature": 0.1 
        }
    )
    print(f"Full response: \n {llm_response}")

    answer = llm_response['choices'][0]['message']['content']

    print(f" \n Response for end user: \n {answer}")